# 1.Setup

In [ ]:
from ri_chat.modeling.utils.chroma_collection import get_chroma_collection
from ri_chat.modeling.embeddings import generate_embeddings, load_embedding_model
import pandas as pd

2026-11-11 12:04:27.564 | INFO     | ri_chat.config:<module>:11 - PROJ_ROOT path is: /home/gabriel_santiago/Estudos/Portfolio/ri_chat


Total de chunks indexados: 6831
IDs: ['1T2024_completas_p2_c0', '1T2024_completas_p5_c0']
Metadados: [{'periodo': '1T2024', 'page_number': 2, 'tipo_doc': 'completas', 'ano': 2024, 'chunk_index': 0, 'trimestre': 1}, {'periodo': '1T2024', 'trimestre': 1, 'ano': 2024, 'page_number': 5, 'chunk_index': 0, 'tipo_doc': 'completas'}]


# 2.Validações

## 2.1.Checando a Coleção

In [ ]:
# checando a colecao
collection = get_chroma_collection()
print(f"Total de chunks indexados: {collection.count()}")
# O esperado é exatamente 6831

Total de chunks indexados: 6831


In [11]:
# checa os 10 primeiros registros salvos no banco
amostra = collection.peek(limit=10)
print("IDs:", amostra["ids"])
print("Metadados:")
pd.DataFrame(amostra["metadatas"]).sort_values('chunk_index')

IDs: ['1T2024_completas_p2_c0', '1T2024_completas_p5_c0', '1T2024_completas_p5_c1', '1T2024_completas_p5_c2', '1T2024_completas_p5_c3', '1T2024_completas_p5_c4', '1T2024_completas_p5_c5', '1T2024_completas_p6_c0', '1T2024_completas_p6_c1', '1T2024_completas_p6_c2']
Metadados:


,chunk_index,periodo,ano,page_number,trimestre,tipo_doc
0,0,1T2024,2024,2,1,completas
1,0,1T2024,2024,5,1,completas
7,0,1T2024,2024,6,1,completas
2,1,1T2024,2024,5,1,completas
8,1,1T2024,2024,6,1,completas
3,2,1T2024,2024,5,1,completas
9,2,1T2024,2024,6,1,completas
4,3,1T2024,2024,5,1,completas
5,4,1T2024,2024,5,1,completas
6,5,1T2024,2024,5,1,completas


## 2.2.Validando a Busca

In [12]:
model = load_embedding_model()

2026-11-11 12:09:50.897 | INFO     | ri_chat.modeling.embeddings:load_embedding_model:18 - Loading embedding model 'BAAI/bge-m3' on 'cuda'...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [15]:
# testando com query
periodo = '4T2024'
query = f"Qual foi o lucro líquido recorrente e o ROE em {periodo}?"
query_vec = generate_embeddings(query, model, show_progress=False)

# buscando por um trimestre
results = collection.query(
    query_embeddings=[query_vec.tolist()],
    n_results=3,
    where={"periodo": f"{periodo}"},
)

In [16]:
# checando os resultados
for i in range(len(results["ids"][0])):
    chunk_id = results["ids"][0][i]
    dist = results["distances"][0][i]
    meta = results["metadatas"][0][i]
    doc = results["documents"][0][i]
    print(f"#{i+1} | ID: {chunk_id} | Sim: {1 - dist:.4f} | Pág: {meta['page_number']}")
    print(doc[:300], "\n" + "-" * 60)

#1 | ID: 4T2024_completas_p8_c8 | Sim: 0.6558 | Pág: 8
[Relatório Itaú Unibanco | Período: 4T2024 | Demonstrações: completas | Página: 8]
###### **grandes números** 

###### **resultado recorrente gerencial** 

###### R$ 10,9 bi **+2,0%** 4T24 4T24 x 3T24 **carteira de crédito** R$ 1.359,1 bi **+6,3%** 4T24 4T24 x 3T24 **margem com clientes** R$ 28,5 bi 
------------------------------------------------------------
#2 | ID: 4T2024_completas_p8_c5 | Sim: 0.6058 | Pág: 8
[Relatório Itaú Unibanco | Período: 4T2024 | Demonstrações: completas | Página: 8]
Em 2024, o resultado recorrente gerencial atingiu R$ 41,4 bilhões, crescimento de 16,2% (excluindo o resultado da Argentina em 2023, o crescimento foi de 18,2%), e o retorno recorrente gerencial foi 1,2 p.p. maior, at 
------------------------------------------------------------
#3 | ID: 4T2024_completas_p39_c1 | Sim: 0.5996 | Pág: 39
[Relatório Itaú Unibanco | Período: 4T2024 | Demonstrações: completas | Página: 39]
Veja mais sobre nossos 